# NEXUS LLM — Module 4: Next Token Prediction Lab

Complete notebook for inspecting DistilGPT2 next-token predictions.

Topics: tokenization, logits, Softmax, top predictions, greedy decoding, temperature, Top-K, Top-P, sampling, visualization and one autoregressive step.

Run cells from top to bottom.

## 1. Imports and configuration

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilbert/distilgpt2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


## 2. Load tokenizer and causal language model

`AutoModelForCausalLM` includes the language-modeling head needed to produce vocabulary logits.

In [2]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Model loaded")
print("Vocabulary size:", model.config.vocab_size)
print("Hidden size:", model.config.n_embd)
print("Transformer layers:", model.config.n_layer)
print("Attention heads:", model.config.n_head)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded
Vocabulary size: 50257
Hidden size: 768
Transformer layers: 6
Attention heads: 12


## 3. Tokenize input text

The model receives token IDs, not raw text.

In [3]:
text = "Artificial intelligence is"
inputs = tokenizer(text, return_tensors="pt")
inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
input_ids = inputs["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(input_ids[0].tolist())

print("Input:", text)
print("Tokens:", tokens)
print("Token IDs:", input_ids)
print("Input shape:", tuple(input_ids.shape))

Input: Artificial intelligence is
Tokens: ['Art', 'ificial', 'Ġintelligence', 'Ġis']
Token IDs: tensor([[8001, 9542, 4430,  318]])
Input shape: (1, 4)


## 4. Extract logits

Logits have shape `(batch, sequence_length, vocabulary_size)`. The final input position produces the scores for the next token.

In [4]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
next_token_logits = logits[:, -1, :]
print("Logits shape:", tuple(logits.shape))
print("Next-token logits shape:", tuple(next_token_logits.shape))

Logits shape: (1, 4, 50257)
Next-token logits shape: (1, 50257)


## 5. Softmax probabilities

Softmax converts raw logits into a probability distribution across the vocabulary.

In [5]:
probabilities = F.softmax(next_token_logits, dim=-1)
print("Probability shape:", tuple(probabilities.shape))
print("Probability sum:", probabilities.sum(dim=-1).item())

Probability shape: (1, 50257)
Probability sum: 1.000011682510376


## 6. Top predicted tokens

In [6]:
def get_top_predictions(probability_tensor, tokenizer, top_n=10):
    top_n = min(top_n, probability_tensor.shape[-1])
    top_probs, top_ids = torch.topk(probability_tensor[0], k=top_n)
    rows = []
    for token_id, probability in zip(top_ids.tolist(), top_probs.tolist()):
        rows.append({
            "token_id": token_id,
            "token": repr(tokenizer.decode([token_id])),
            "probability": probability,
            "probability_percent": probability * 100
        })
    return pd.DataFrame(rows)

prediction_df = get_top_predictions(probabilities, tokenizer, top_n=10)
display(prediction_df)

,token_id,token,probability,probability_percent
0,257,' a',0.164362,16.436212
1,262,' the',0.066132,6.613249
2,407,' not',0.048673,4.867272
3,530,' one',0.036778,3.677796
4,281,' an',0.035226,3.522589
5,783,' now',0.016249,1.624936
6,5033,' becoming',0.015808,1.580811
7,635,' also',0.014690,1.469033
8,655,' just',0.011326,1.132585
9,991,' still',0.011237,1.123651
